# Diabetes Scenario 3 — How was diabetes treated in hospital, and did treatment follow the glucose results?

_Describing practice patterns with many categorical variables. Level: third analysis. About 90 minutes._

**Before you start.** These are 101,766 inpatient encounters of 71,518 patients with a diabetes
diagnosis in 130 US hospitals, 1999 to 2008, from an electronic health record warehouse: nobody
was randomised, HbA1c was rarely measured, several drug classes of today did not exist, and a
readmission to a hospital outside the network is invisible. One row is **one encounter**; a
patient with several stays appears several times, so the analyses keep the first encounter of
each patient and drop the encounters that ended in death or hospice, as the published analysis
did. Codes: 0 = No, 1 = Yes; Gender 1 = Male, 2 = Female; age only as 10-year bands.

**How this notebook works.** Every analysis is a *federated* call: the code selects variables,
defines a cohort with filters and asks the MIP for aggregates (counts, means, a test, a model). No
encounter record ever reaches this notebook. Run the cells in order; each step says what to look
for and what result to expect.

**Learning goals.** Summarise many variables at once and decide which matter; read a four-level
categorical variable (not prescribed, steady, increased, decreased) as both a prevalence and a
change indicator; relate management to a test result with a cross-tabulation on an ordered
exposure; and recognise historical prescribing patterns and their limits.

**Clinical background.** For each of 21 diabetes drugs the record says whether it was given
during the stay and, if so, whether the dose was kept, increased or decreased. Two summary
variables say whether any diabetes drug was given and whether any change was made. Two test
results are available as ranges: HbA1c (long-term control) and the highest serum glucose of the
stay (short-term). Good inpatient care would adjust treatment when either is high. This scenario
describes what was actually done, in 130 US hospitals between 1999 and 2008: insulin and
sulfonylureas dominate, metformin is less used than today, rosiglitazone and pioglitazone were
common, and the classes that now define diabetes care did not exist. Only drugs administered
during the stay are known; home medication and discharge prescriptions are not.

## 1. Connect to the MIP and open the Diabetes pathology

In [ ]:
import mip
from mip.filters import F

client = mip.Client.from_env()          # connection configured by the MIP portal
catalog = client.catalog()

try:
    dm = catalog.data_model("Diabetes")    # add version="DiabetesUCI" if the catalogue holds several versions
except LookupError:
    display(catalog.summaries())           # the pathology label differs: pick it from this list
    raise
ds = dm.datasets.list()[0]              # one dataset: Diabetes 130-US hospitals 1999-2008
display(dm)
display(ds)

In [ ]:
# The variable tree follows the encounter: Encounter, Admission, Demographics, Diagnoses
# (primary, secondary, summary), Hospital care, Prior utilisation, Laboratory, Diabetes
# medication (individual drugs, summary), Discharge, Outcome.
dm.tree()

## 2. Toolbox

In [ ]:
# ---- Toolbox: small helpers that turn MIP results into readable tables and charts. ----
# Run this cell once. You do not need to read it to follow the scenario; come back to it
# when you want to see how a result payload is unpacked.
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from mip.preprocessing import MissingValuesHandler

POOLED = "all datasets"                 # the row describe() adds across the datasets of the pathology
PCT = "{:.1f}"                          # number format of the percentage columns


def analysis(variables, filters=None, drop_missing=False):
    """A Pipeline on the Diabetes 130 dataset with the given variables and cohort filter.
    Tests and models need drop_missing=True: the platform requires an explicit missing-values
    step and analyses complete cases only. describe() keeps it off so it can count missing values."""
    handler = MissingValuesHandler(strategies={v: "drop" for v in variables}) if drop_missing else None
    return mip.Pipeline(analysis_set=dm.select(datasets=[ds], variables=variables), filters=filters,
                        handle_missing=handler)


def labels_of(variable):
    """{code: label} of a categorical variable, whatever format the catalogue uses."""
    raw = variable._data.get("enumerations") or {}
    if isinstance(raw, dict):
        return {str(k): str(v) for k, v in raw.items()}
    return {str(item.get("code") if isinstance(item, dict) else item):
            str(item.get("label") if isinstance(item, dict) else item) for item in raw}


def rows_for(result, variable):
    """The pooled 'describe' row of one variable (the platform adds one row per dataset as well)."""
    return [r for r in result.raw.get("featurewise", [])
            if r.get("variable") == variable._code and r.get("dataset") == POOLED]


def numeric_summary(result, variables):
    """n, missing, mean, SD, quartiles and range of numerical variables from a describe result."""
    rows = []
    for v in variables:
        for r in rows_for(result, v):
            d = r.get("data") or {}
            rows.append({"variable": v.label, "n": d.get("num_dtps"), "missing": d.get("num_na"),
                         "mean": d.get("mean"), "sd": d.get("std"), "q1": d.get("q1"), "median": d.get("q2"),
                         "q3": d.get("q3"), "min": d.get("min"), "max": d.get("max")})
    return pd.DataFrame(rows)


def category_counts(result, variable):
    """Counts and percentages of the levels of a categorical variable from a describe result."""
    lab = labels_of(variable)
    rows = []
    for r in rows_for(result, variable):
        counts = (r.get("data") or {}).get("counts") or {}
        total = sum(counts.values())
        for c, n in counts.items():
            rows.append({"variable": variable.label, "level": lab.get(str(c), str(c)), "count": n,
                         "percent": 100 * n / total if total else np.nan})
    return pd.DataFrame(rows)


def level_percent(result, variable, levels=("Yes",)):
    """Percentage of the given levels (labels) of a categorical variable in a describe result."""
    k = category_counts(result, variable)
    return float(k.loc[k["level"].isin(levels), "percent"].sum()) if len(k) else np.nan


def by_level(numeric, group, levels=None, base_filter=None):
    """n, mean, median and quartiles of a numerical variable in each level of a categorical one
    (one describe per level). levels: the codes to use, by default every code of the variable."""
    lab = labels_of(group)
    rows = []
    for c in levels or lab:
        f = F(group) == c
        if base_filter is not None:
            f = f & base_filter
        res = analysis([numeric], f).describe()
        for r in rows_for(res, numeric):
            d = r.get("data") or {}
            rows.append({group.label: lab.get(c, c), "n": d.get("num_dtps"), "mean": d.get("mean"),
                         "q1": d.get("q1"), "median": d.get("q2"), "q3": d.get("q3")})
    return pd.DataFrame(rows)


def crosstab(exposure, outcome, base_filter=None, event="1", levels=None, event_label=None):
    """Cross-tabulation of outcome by exposure, as the MIP does it: counts come from one
    describe per exposure level, the p-value from the Chi-squared Test on the whole cohort.
    `event` is the outcome code counted in the percentage column, or a list of codes."""
    lab_x, lab_y = labels_of(exposure), labels_of(outcome)
    events = [event] if isinstance(event, str) else list(event)
    pct_col = "% " + (event_label or lab_y.get(events[0], events[0]))
    rows = []
    for c in levels or lab_x:
        f = (F(exposure) == c) & F(outcome).is_not_null()
        if base_filter is not None:
            f = f & base_filter
        res = analysis([exposure, outcome], f).describe()
        counts = {}
        for r in rows_for(res, outcome):
            counts = (r.get("data") or {}).get("counts") or {}
        n = sum(counts.values())
        if n == 0:
            continue
        row = {exposure.label: lab_x.get(c, c), "n": n}
        for yc, ylabel in lab_y.items():
            row[ylabel] = counts.get(yc, 0)
        row[pct_col] = 100 * sum(counts.get(e, 0) for e in events) / n
        rows.append(row)
    table = pd.DataFrame(rows)
    f = F(exposure).is_not_null() & F(outcome).is_not_null()
    if levels:
        f = f & F(exposure).isin(list(levels))
    if base_filter is not None:
        f = f & base_filter
    chi = analysis([exposure, outcome], f, drop_missing=True).chi_square_test(x=exposure, y=outcome)
    stats = chi.raw if isinstance(chi.raw, dict) else {}
    table.attrs["chi2"] = stats.get("chi2")
    table.attrs["p"] = stats.get("p_value", stats.get("p"))
    table.attrs["outcome"] = outcome.label
    table.attrs["outcome_levels"] = list(lab_y.values())
    table.attrs["events"] = [lab_y.get(e, e) for e in events]
    return table


def collapse(table, groups):
    """Pool the rows of a crosstab into two groups ({name: [row labels]}) and test the 2x2 table
    with a chi-square with Yates' correction, as the platform does, from the aggregate counts."""
    first = table.columns[0]
    levels, events = table.attrs["outcome_levels"], table.attrs["events"]
    rows = []
    for name, members in groups.items():
        part = table[table[first].isin(members)]
        counts = {lev: int(part[lev].sum()) for lev in levels}
        n = sum(counts.values())
        hit = sum(counts[e] for e in events)
        rows.append({first: name, "n": n, **counts, "% " + "/".join(events): 100 * hit / n if n else np.nan})
    out = pd.DataFrame(rows)
    (a, b), (c, d) = [(r["n"] - sum(r[e] for e in events), sum(r[e] for e in events)) for r in rows[:2]]
    n = a + b + c + d
    chi2 = n * (abs(a * d - b * c) - n / 2) ** 2 / ((a + b) * (c + d) * (a + c) * (b + d))
    out.attrs["chi2"], out.attrs["p"] = chi2, math.erfc(math.sqrt(chi2 / 2))
    return out


def band_rates(outcomes, bands, base_filter=None, event="1"):
    """% of encounters with each outcome in bands of a variable: bands maps a label to a filter."""
    rows = []
    for label, f in bands.items():
        if base_filter is not None:
            f = f & base_filter
        res = analysis(outcomes, f).describe()
        row = {"band": label}
        for y in outcomes:
            counts = {}
            for r in rows_for(res, y):
                counts = (r.get("data") or {}).get("counts") or {}
            n = sum(counts.values())
            row.setdefault("n", n)
            row[y.label] = 100 * counts.get(event, 0) / n if n else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "n/a"
    return "< 0.001" if p < 0.001 else f"{p:.3f}"


def show_crosstab(table, title=None):
    """Display a crosstab table, its chi-square test and a bar chart of the percentages."""
    pct_col = [c for c in table.columns if c.startswith("% ")][0]
    display(table.style.format({pct_col: PCT}).hide(axis="index"))
    print(f"Chi-squared Test: chi2 = {table.attrs['chi2']:.1f}, p = {fmt_p(table.attrs['p'])}")
    ax = table.set_index(table.columns[0])[pct_col].plot(kind="bar", color="#4C72B0", figsize=(6, 3))
    ax.set_ylabel(pct_col)
    ax.set_title(title or f"{table.attrs['outcome']} by {table.columns[0]}")
    ax.set_xlabel("")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


def ttest_summary(result):
    """Independent t-test result -> t, p, df, mean difference with its 95% CI, Cohen's d, n."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    return {k: raw.get(k) for k in ("t_stat", "p", "df", "mean_diff", "ci_lower", "ci_upper", "cohens_d", "n_obs")}


def odds_ratio_table(result, variables):
    """Logistic regression result -> one row per term with odds ratio, 95% CI and p.
    Categorical covariates appear as 'variable = level' relative to the reference level
    (the first code of the variable) that the platform leaves out."""
    by_code = {v._code: v for v in variables}

    def pretty(term):
        term = str(term)
        if "[" in term and term.endswith("]"):
            base, level = term[:-1].split("[", 1)
            v = by_code.get(base)
            return f"{v.label} = {labels_of(v).get(level, level)}" if v else term
        v = by_code.get(term)
        return f"{v.label} (per unit)" if v else term

    frame = result.to_frame().copy()
    frame = frame[~frame["feature"].astype(str).str.lower().isin(["intercept", "const"])]
    frame["term"] = frame["feature"].map(pretty)
    for col in ("odds_ratio", "or_lower_ci", "or_upper_ci", "p"):
        if col not in frame.columns:
            frame[col] = np.nan
    out = frame[["term", "odds_ratio", "or_lower_ci", "or_upper_ci", "p"]]
    return out.rename(columns={"or_lower_ci": "ci_low", "or_upper_ci": "ci_high"}).reset_index(drop=True)


def show_odds_ratios(table, title):
    """Table and forest plot (log scale) of odds ratios."""
    display(table.style.format({"odds_ratio": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": "{:.4f}"}).hide(axis="index"))
    t = table.iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(t) + 1.5))
    ax.errorbar(t["odds_ratio"], range(len(t)),
                xerr=[t["odds_ratio"] - t["ci_low"], t["ci_high"] - t["odds_ratio"]], fmt="o", capsize=4)
    ax.axvline(1, linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(t["term"])
    ax.set_xlabel("odds ratio (log scale)")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


print("Toolbox ready.")

## 3. The variables of this scenario

| Code | Label in the MIP | Tree location (under Diabetes) | Role | Values |
|---|---|---|---|---|
| `first_encounter` | First encounter of the patient | Encounter | cohort filter: one encounter per patient; curation-derived | 0 No; 1 Yes |
| `death_or_hospice` | Died or discharged to hospice | Discharge | cohort filter: encounters that could be readmitted; curation-derived | 0 No; 1 Yes |
| `diabetesMed` | Any diabetes medication | Diabetes medication / Summary | summary: any diabetes drug | 0 No; 1 Yes |
| `change` | Change in diabetes medication | Diabetes medication / Summary | summary: any change | 0 No change; 1 Change in diabetes medication |
| `insulin` | Insulin | Diabetes medication / Individual drugs | drug, 4 levels | 0 Not prescribed; 1 Steady dose; 2 Dose increased; 3 Dose decreased |
| `metformin` | Metformin | Diabetes medication / Individual drugs | drug, 4 levels | 0 Not prescribed; 1 Steady dose; 2 Dose increased; 3 Dose decreased |
| `glipizide` | Glipizide | Diabetes medication / Individual drugs | drug, 4 levels | 0 Not prescribed; 1 Steady dose; 2 Dose increased; 3 Dose decreased |
| `glyburide` | Glyburide | Diabetes medication / Individual drugs | drug, 4 levels | 0 Not prescribed; 1 Steady dose; 2 Dose increased; 3 Dose decreased |
| `glimepiride` | Glimepiride | Diabetes medication / Individual drugs | drug, 4 levels | 0 Not prescribed; 1 Steady dose; 2 Dose increased; 3 Dose decreased |
| `pioglitazone` | Pioglitazone | Diabetes medication / Individual drugs | drug, 4 levels | 0 Not prescribed; 1 Steady dose; 2 Dose increased; 3 Dose decreased |
| `rosiglitazone` | Rosiglitazone | Diabetes medication / Individual drugs | drug, 4 levels | 0 Not prescribed; 1 Steady dose; 2 Dose increased; 3 Dose decreased |
| `repaglinide` | Repaglinide | Diabetes medication / Individual drugs | drug, 4 levels | 0 Not prescribed; 1 Steady dose; 2 Dose increased; 3 Dose decreased |
| `A1Cresult` | HbA1c result | Laboratory | test result (Step 4) | 0 Not measured; 1 Normal (below 7%); 2 Above 7% (7 to 8%); 3 Above 8% |
| `max_glu_serum` | Maximum serum glucose | Laboratory | test result (Step 5) | 0 Not measured; 1 Normal; 2 Above 200 mg/dL; 3 Above 300 mg/dL |
| `age_group3` | Age group (3 levels) | Demographics | who received insulin; curation-derived | 1 Under 30 years; 2 30-59 years; 3 60 years and over |
| `diag_1_grp` | Primary diagnosis group | Diagnoses / Primary diagnosis | who received insulin; curation-derived | 9 codes, listed in the tree |
| `readmit_30` | Readmitted within 30 days | Outcome | outcome of Step 6; curation-derived | 0 No; 1 Yes |
| `num_medications` | Number of medications | Hospital care | complexity (Step 6) | integer, 0-100 |
| `hba1c_group` | HbA1c group (published analysis) | Laboratory | extension; curation-derived | 1 Not measured; 2 Measured, result below 8%; 3 Measured, above 8%, diabetes medication changed; 4 Measured, above 8%, diabetes medication not changed |

Variables are picked by their **label**, as the tree shows it; the tree location tells you where
to find them in `dm.tree()` and in the MIP interface. The code (for example `A1Cresult`) is what
the csv column is called; you will see it in some result payloads.

In [ ]:
any_med   = dm.variables["Any diabetes medication"]
change    = dm.variables["Change in diabetes medication"]
drugs = [dm.variables[name] for name in ("Insulin", "Metformin", "Glipizide", "Glyburide", "Glimepiride",
                                          "Pioglitazone", "Rosiglitazone", "Repaglinide")]
insulin, metformin = drugs[0], drugs[1]
a1c       = dm.variables["HbA1c result"]
glucose   = dm.variables["Maximum serum glucose"]
age3      = dm.variables["Age group (3 levels)"]
diag      = dm.variables["Primary diagnosis group"]
readmit   = dm.variables["Readmitted within 30 days"]
meds      = dm.variables["Number of medications"]
display(mip.to_frame([any_med, change] + drugs + [a1c, glucose, age3, diag, readmit, meds]))

## Step 1. Which drugs were used?

The cohort of Scenarios 1 and 2 (first encounter, no death or hospice: 69,973 encounters). One
`describe()` of the eight drugs and the two summary variables; for each drug the three
prescribed levels (steady, increased, decreased) are added up.

*Expected (percentage of encounters in which the drug was given):* insulin 51%, metformin 21%,
glipizide 13%, glyburide 11%, pioglitazone 7.5%, rosiglitazone 6.7%, glimepiride 5.3%,
repaglinide 1.3%. Any diabetes drug 76%; a change in diabetes medication in 45% of all
encounters, 59% of those with any diabetes drug. Insulin was the workhorse of inpatient diabetes
care; metformin, today's first-line drug, was given to one patient in five. The remaining 13
drugs were each given to fewer than 1%.

In [ ]:
first = dm.variables["First encounter of the patient"]
death = dm.variables["Died or discharged to hospice"]
cohort = (F(first) == "1") & (F(death) == "0")      # one encounter per patient, able to be readmitted

GIVEN = ("Steady dose", "Dose increased", "Dose decreased")
practice = analysis(drugs + [any_med, change], cohort).describe()
given = pd.Series({v.label: level_percent(practice, v, GIVEN) for v in drugs}).sort_values(ascending=False)
display(given.to_frame("% of encounters given the drug").style.format(PCT))
print(f"Any diabetes drug: {level_percent(practice, any_med):.1f}%; medication changed: "
      f"{level_percent(practice, change, ('Change in diabetes medication',)):.1f}% of all encounters")
on_drug = analysis([change], cohort & (F(any_med) == "1")).describe()
print(f"Medication changed among encounters with any diabetes drug: {level_percent(on_drug, change, ('Change in diabetes medication',)):.1f}%")
ax = given.iloc[::-1].plot(kind="barh", color="#4C72B0", figsize=(7, 3.5))
ax.set_xlabel("% of encounters"); ax.set_title("Diabetes drugs given during the stay, 1999 to 2008")
plt.tight_layout()
plt.show()

## Step 2. Insulin: steady, up or down?

The four levels of insulin, from the same `describe()`.

*Expected:* not given 49.0%; steady 30.9%; increased 9.7%; decreased 10.5%. When insulin was
given, its dose was changed in about 40% of stays, as often down as up.

In [ ]:
levels = category_counts(practice, insulin)
display(levels.style.format({"percent": PCT}).hide(axis="index"))
given_insulin = levels[levels["level"] != "Not prescribed"]
changed = given_insulin[given_insulin["level"] != "Steady dose"]["count"].sum()
print(f"Dose changed in {100 * changed / given_insulin['count'].sum():.0f}% of the stays with insulin")

## Step 3. Who received insulin?

The cross-tabulation of insulin (given: any of the three prescribed levels) by age group, then by
primary diagnosis group; metformin by age group for comparison.

*Expected:* insulin given in 75% of patients under 30, 54% at 30 to 59, 49% at 60 and over; in
75% of patients admitted primarily for diabetes versus 49% of the others. Metformin shows the
opposite age pattern: 10%, 25%, 20%. Young diabetic inpatients are mostly type 1
(insulin-dependent); elderly patients with other primary diagnoses often had their diabetes
managed lightly during the stay.

In [ ]:
GIVEN_CODES = ["1", "2", "3"]
insulin_age = crosstab(age3, insulin, base_filter=cohort, event=GIVEN_CODES, event_label="given")
metformin_age = crosstab(age3, metformin, base_filter=cohort, event=GIVEN_CODES, event_label="given")
by_age = pd.DataFrame({"insulin given": insulin_age.set_index(age3.label)["% given"],
                       "metformin given": metformin_age.set_index(age3.label)["% given"]})
display(by_age.style.format(PCT))

insulin_diag = crosstab(diag, insulin, base_filter=cohort, event=GIVEN_CODES, event_label="given")
display(insulin_diag[[diag.label, "n", "% given"]].style.format({"% given": PCT}).hide(axis="index"))
print(f"Chi-squared Test: chi2 = {insulin_diag.attrs['chi2']:.1f}, p = {fmt_p(insulin_diag.attrs['p'])}")
others = analysis([insulin], cohort & (F(diag) != "4")).describe()
print(f"Insulin given when the primary diagnosis is not diabetes: {level_percent(others, insulin, GIVEN):.1f}%")

short = insulin_diag[diag.label].str.replace(r" \(.*\)", "", regex=True)        # drop the ICD-9 ranges
ax = pd.Series(insulin_diag["% given"].values, index=short).iloc[::-1].plot(kind="barh", color="#4C72B0", figsize=(7, 3.6))
ax.set_xlabel("% of encounters given insulin"); ax.set_title("Insulin given, by primary diagnosis group")
plt.tight_layout()
plt.show()

## Step 4. Did treatment respond to HbA1c?

Two cross-tabulations by HbA1c result: any change in diabetes medication, and an insulin dose
increase.

*Expected:* medication changed in 42.7% when HbA1c was not measured, 43.0% when below 7%, 49.4%
when 7 to 8%, 65.0% when above 8% (p < 0.001). Insulin dose increased in 8.6%, 9.4%, 10.6% and
19.5% respectively. A clear gradient: a high HbA1c doubled the chance of an insulin increase. But
35% of patients with HbA1c above 8% had no change at all.

In [ ]:
show_crosstab(crosstab(a1c, change, base_filter=cohort, event_label="changed"), "Diabetes medication changed, by HbA1c result")
show_crosstab(crosstab(a1c, insulin, base_filter=cohort, event="2", event_label="insulin increased"), "Insulin dose increased, by HbA1c result")

## Step 5. Did treatment respond to the glucose of the stay?

The cross-tabulation of a medication change by the maximum serum glucose of the stay.

*Expected:* change in 45.2% when glucose was not measured (or not flagged), 30.1% when normal,
46.3% when above 200 mg/dL, 62.6% when above 300 mg/dL (p < 0.001). Again a gradient, again
incomplete: more than one third of patients with a glucose above 300 left with unchanged
diabetes medication.

In [ ]:
show_crosstab(crosstab(glucose, change, base_filter=cohort, event_label="changed"), "Diabetes medication changed, by maximum serum glucose")

## Step 6. Does changing medication predict readmission?

Readmission cross-tabulated by a medication change and by any diabetes drug; then the number of
medications per group, the trap behind the result.

*Expected:* readmission 9.4% with a change versus 8.6% without (p = 0.0001); 9.4% with any
diabetes drug versus 7.5% without (p < 0.001). Patients whose treatment was changed were
readmitted slightly *more* often. Before concluding that changing treatment is harmful, look at
the number of medications: 17.9 on average when treatment was changed versus 13.8 when not.
Treatment is changed in sicker, more complex patients. This is confounding by indication, the
central trap of observational treatment comparisons.

In [ ]:
show_crosstab(crosstab(change, readmit, base_filter=cohort, event_label="readmitted"), "Readmission, by change in diabetes medication")
show_crosstab(crosstab(any_med, readmit, base_filter=cohort, event_label="readmitted"), "Readmission, by any diabetes medication")
display(by_level(meds, change, base_filter=cohort).style.format({c: "{:.1f}" for c in ("mean", "q1", "median", "q3")}).hide(axis="index"))

## Step 7. State the result

A model answer:

> In 69,973 hospital encounters of patients with diabetes (US, 1999 to 2008), three quarters
> received a diabetes drug, insulin in half of all encounters and metformin in a fifth. Diabetes
> medication was changed in 45% of stays. Changes did follow the laboratory results, with
> medication changed in 65% of stays with HbA1c above 8% and 63% with glucose above 300 mg/dL,
> against about 43% otherwise, but a third of poorly controlled patients left with unchanged
> treatment. Patients whose treatment was changed were readmitted slightly more often, which
> reflects their greater complexity rather than an effect of the change.

**Questions for discussion.** Sulfonylureas were given to almost 30% of patients and metformin
to 21%: how does this compare with current guidelines, and why might metformin be stopped during
a hospital stay even today? Insulin was decreased as often as increased: give two clinical
reasons for decreasing insulin in hospital. How would you design a study to know whether
changing diabetes medication in hospital reduces readmission? Is a decrease a "change in the
right direction" for a patient with HbA1c above 8%, and what does this say about summary
variables?

**Optional extension.** Reproduce the paper's four HbA1c groups: cross-tabulate the HbA1c result
against the HbA1c group and check that the counts agree with the rule in the variable's
methodology note (above 8% is split by whether the medication was changed). Then discuss why the
paper merged "below 7%" and "7 to 8%" into one group.

In [ ]:
hba1c_grp = dm.variables["HbA1c group (published analysis)"]
rule = crosstab(a1c, hba1c_grp, base_filter=cohort)
display(rule.drop(columns=[c for c in rule.columns if c.startswith("% ")]).style.hide(axis="index"))